# Application: Minimax combinations of VAR and LP estimators (Section 5)

This notebook reproduces Figures 9-13 of the paper. For each of the four applications it:

1. loads the VAR and LP estimates (sourced from Montiel Olea, Plagborg-Møller, Qian & Wolf replication files);
2. computes the application-specific bound m for a given misspecification share;
3. runs SMA at that m;
4. constructs the bias-corrected VAR estimator horizon by horizon;
5. compares its worst-case risk to the minimax-optimal linear combination of the LP and the VAR estimators.

The default misspecification share is M = 1% (Table 1 in the paper). 

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import pickle

from scripts.application import (load_application_data, build_applications_dict, run_or_load_application_sma, estimate_bnm_worst_case_risk, compute_application_metrics, gk_cumulative_diff_1sd)
from scripts.plotting import (plot_irf_panel, plot_application_hausman, plot_application_bias_correction,plot_application_corrected_irfs, plot_application_risk_improvement)


captions = {"GK": "Gertler & Karadi (2015)", "RR": "Romer & Romer (2010)", "R": "Ramey (2011)", "FORD": "Francis, Owyang, Roush, and DiCecio (2014)"}


root_folder = Path.cwd().parent
data_path = root_folder / 'data'
irfs_data_path = data_path / 'res_application.mat'

## Load the VAR and LP estimates

### Upload four applications from the paper (Note: Gertler and Karadi (2015) is re-estimated here from the raw data)

res_application.mat only stores GK in log levels to a 1-UNIT shock. Here we re-estimate this applicaton from the raw data (Monetarydat.xlsx) to make it cumulative first deifferences.

In [ ]:
misspec_share = 0.01 

raw_data = load_application_data(str(irfs_data_path))

raw_data["GK"] = gk_cumulative_diff_1sd(data_path / "Monetarydat.xlsx")

applications = build_applications_dict(raw_data, misspec_share)

m_table = pd.DataFrame([
    {
        "application": key,
        "description": app["caption"],
        "T": app["T"],
        "misspec_share": misspec_share,
        "m = sqrt(T*Mcal)": app["m"],
    }
    for key, app in applications.items()
])
print(m_table.to_string(index=False))

## LP vs VAR impulse responses

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=False)

plot_irf_panel(axes[0, 0], raw_data["GK"]["VAR"], raw_data["GK"]["VAR_se"],raw_data["GK"]["LP"], raw_data["GK"]["LP_se"], "Monetary policy shock to industrial production", captions["GK"], xtick_step=5)
plot_irf_panel(axes[0, 1], raw_data["RR"]["VAR"], raw_data["RR"]["VAR_se"], raw_data["RR"]["LP"], raw_data["RR"]["LP_se"],"Tax shock to GDP", captions["RR"], xtick_step=2)
plot_irf_panel(axes[1, 0], raw_data["R"]["VAR"], raw_data["R"]["VAR_se"], raw_data["R"]["LP"], raw_data["R"]["LP_se"], "Military news shock to GDP", captions["R"], xtick_step=2)
plot_irf_panel(axes[1, 1], raw_data["FORD"]["VAR"], raw_data["FORD"]["VAR_se"], raw_data["FORD"]["LP"], raw_data["FORD"]["LP_se"], "TFP shock to labor productivity", captions["FORD"], xtick_step=2)

legend_handles = [
    Line2D([0], [0], color='navy', label='VAR'),
    Line2D([0], [0], color='dimgray', label='LP')]

fig.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=True, bbox_to_anchor=(0.5, 0.03), borderaxespad=0.2)

plt.subplots_adjust(bottom=0.15, hspace=0.38, wspace=0.42)
plt.show()

## Run SMA for each application

In [ ]:
saved_sma_app = data_path / f"application_sma_results_M{int(round(10000 * misspec_share)):04d}bp.pkl"
rerun_sma = False

application_sma = run_or_load_application_sma(applications, cache_path=str(saved_sma_app), force_rerun=rerun_sma, n_samples_risk=1000, seed_base=123, show_progress=True)

## Worst-case risk for each application

In [ ]:
saved_sma_risk = data_path / f"application_bnm_wc_risks_M{int(round(10000 * misspec_share)):04d}bp.pkl"
rerun_risks = False

if (not rerun_risks) and saved_sma_risk.exists():
    with open(saved_sma_risk, "rb") as f:
        bnm_wc = pickle.load(f)
    print(f"Loaded normalized BNM risk estimates from {saved_sma_risk}")
else:
    bnm_wc = {}
    for j, (key, res) in enumerate(application_sma.items()):
        m = float(application_sma[key]["m"])
        bnm_wc[key] = estimate_bnm_worst_case_risk(
            theta_grid=res["theta_grid"],
            avg_prior=res["avg_prior"],
            m=m,
            n_theta=301,
            n_mc=1000,
            seed=54321 + j)
        print(f"{key}: m={m:.4f}, "f"normalized SMA worst-case risk={bnm_wc[key]['wc_risk']:.4f}, "f"linear normalized risk={m**2/(1+m**2):.4f}")

    with open(saved_sma_risk, "wb") as f:
        pickle.dump(bnm_wc, f)

bnm_risk_table = pd.DataFrame([
    {
        "application": key,
        "m": application_sma[key]["m"],
        "SMA_normalized_wc_risk": bnm_wc[key]["wc_risk"],
        "linear_normalized_wc_risk": application_sma[key]["m"]**2 / (1 + application_sma[key]["m"]**2),
        "theta_at_wc": bnm_wc[key]["theta_at_wc"],
    }
    for key in application_sma
])

print(bnm_risk_table.to_string(index=False))

## Horizon-by-horizon metrics

In [ ]:
app_metrics = {}

for key, app in applications.items():
    app_metrics[key] = compute_application_metrics(app=app, sma_res=application_sma[key], normalized_wc_risk=bnm_wc[key]["wc_risk"])

metrics_table = pd.DataFrame([
    {
        "application": key,
        "m": app_metrics[key]["m"],
        "omega*_linear": app_metrics[key]["omega_star"],
        "avg_risk_improvement_pct": np.nanmean(app_metrics[key]["risk_improvement_pct"]),
        "min_risk_improvement_pct": np.nanmin(app_metrics[key]["risk_improvement_pct"]),
        "max_abs_Hausman": np.nanmax(app_metrics[key]["abs_Delta"]),
        "max_abs_bias_correction": np.nanmax(np.abs(app_metrics[key]["bias_correction"])),
    }
    for key in applications
])

print(metrics_table.to_string(index=False))

## Figures 10-13 (for each application)

In [ ]:
# Figure 10: Hausman statistic across horizons
plot_application_hausman(applications, app_metrics)

In [ ]:
# Figure 11: Bias correction across horizons
plot_application_bias_correction(applications, app_metrics)

In [ ]:
# Figure 12: VAR, LP, optimal linear, and bias-corrected VAR
plot_application_corrected_irfs(applications, app_metrics)

In [ ]:
# Figure 13: Risk improvement over optimal linear combination
plot_application_risk_improvement(applications, app_metrics)